In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-07 16:39:19.990492: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-07 16:39:20.730509: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-07 16:39:21,785 [DEBUG] [Rain] Rain is initialized
2023-07-07 16:39:21,787 [DEBUG] [Provisioner] Creating coordinator
2023-07-07 16:39:21,787 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-07 16:39:21,788 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-07 16:39:21,789 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-07 16:39:21,791 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 16:39:21,793 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 16:39:21,794 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-07 16:39:21,808 [INFO] [Provisioner] provisioner is serving
2023-07-07 16:39:21,810 [DEBUG] [Provisioner] Starting coordinator
2023-07-07 16:39:21,811 [INFO] [Coordinator] coordinator is serving
2023-07-07 16:39:21,812 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-07 16:39:21,821 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 16:39:21,823 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-07 16:39:21,827 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisioner
2023-07-07 16:39:21,829 [DEBUG] [Provisioner] Provision requested the coordinator to get the number of workers
2023-07-07 16:39:21,830 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-07 16:39:21,831 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-07 16:39:21,833 [INFO] [Worker_50151] Worker is running 

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 3s 9ms/step - loss: 0.7140 - accuracy: 0.7760
Epoch 2/5
157/157 [==============================] - 3s 9ms/step - loss: 0.7069 - accuracy: 0.7778
Epoch 2/5
157/157 [==============================] - 3s 9ms/step - loss: 0.7104 - accuracy: 0.7771
Epoch 2/5
157/157 [==============================] - 1s 9ms/step - loss: 0.3149 - accuracy: 0.9059
Epoch 3/5
157/157 [==============================] - 1s 9ms/step - loss: 0.3115 - accuracy: 0.9095
Epoch 3/5
157/157 [==============================] - 1s 9ms/step - loss: 0.2286 - accuracy: 0.9306
Epoch 4/5
157/157 [==============================] - 1s 9ms/step - loss: 0.2404 - accuracy: 0.9292
Epoch 4/5
157/157 [==============================] - 1s 9ms/step - loss: 0.1877 - accuracy: 0.9418
Epoch 5/5
157/157 [==============================] - 1s 9ms/step - loss: 0.1916 - accuracy: 0.9428
Epoch 5/5
157/157 [==============================] - 1s 9ms/step - loss: 0.1950 - accurac

2023-07-07 16:39:33,162 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2


sending data to divider
147/157 [===========================>..] - ETA: 0s - loss: 0.1683 - accuracy: 0.9490

2023-07-07 16:39:33,165 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2


...

139/157 [=========================>....] - ETA: 0s - loss: 0.0869 - accuracy: 0.9728

2023-07-07 16:39:49,471 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 3.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 3.


157/157 [==============================] - 1s 9ms/step - loss: 0.0862 - accuracy: 0.9727
sending data to divider


2023-07-07 16:39:49,549 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-07 16:39:49,551 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-07 16:39:49,608 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-07 16:39:49,617 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
DEBUG:DeepLearning:Asynchronous update is done by worker 1
2023-07-07 16:39:49,642 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 1.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 1.
2023-07-07 16:39:52,319 [DEBUG

sending data to divider


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.0762 - accuracy: 0.9792

Test accuracy: 97.9%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-07 16:39:52,852 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-07 16:39:52,853 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-07 16:39:52,855 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-07 16:39:52,857 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-07 16:39:52,860 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 16:39:52,863 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-07 16:39:52,867 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisione

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 3s 9ms/step - loss: 0.1077 - accuracy: 0.9695
Epoch 2/5
Epoch 2/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0936 - accuracy: 0.9705
Epoch 3/5
Epoch 3/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0919 - accuracy: 0.9718
Epoch 3/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0793 - accuracy: 0.9747
Epoch 4/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0804 - accuracy: 0.9754
Epoch 4/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0887 - accuracy: 0.9717
Epoch 4/5
157/157 [==============================] - 2s 10ms/step - loss: 0.0722 - accuracy: 0.9768
Epoch 5/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0679 - accuracy: 0.9782


2023-07-07 16:40:04,267 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-07 16:40:04,270 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-07 16:40:04,271 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-07 16:40:04,273 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3


sending data to divider
sending data to divider
157/157 [==============================] - 1s 9ms/step - loss: 0.0727 - accuracy: 0.9774


DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-07 16:40:04,299 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-07 16:40:04,301 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2


...

2023-07-07 16:40:18,967 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-07 16:40:18,970 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1


sending data to divider
157/157 [==============================] - 1s 7ms/step - loss: 0.0546 - accuracy: 0.9831
sending data to divider


2023-07-07 16:40:18,982 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-07 16:40:18,984 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
2023-07-07 16:40:18,989 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-07 16:40:18,990 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3


sending data to divider


2023-07-07 16:40:19,036 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-07 16:40:19,042 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-07 16:40:19,050 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-07 16:40:19,073 [DEBUG] [DeepLearning] Iteration 3/3 complete.
DEBUG:DeepLearning:Iteration 3/3 complete.
2023-07-07 16:40:19,075 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-07 16:40:19,076 [DEBUG] 

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 1ms/step - loss: 0.0679 - accuracy: 0.9826

Test accuracy: 98.3%


2023-07-07 16:41:08,853 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 16:41:11,025 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
2023-07-07 16:41:11,025 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
INFO:Worker_50151:Running the worker with id: 1 on iteration: 1


Epoch [1/5], Loss: 0.7393, Accuracy: 0.7795
Epoch [2/5], Loss: 0.3133, Accuracy: 0.9050
Epoch [3/5], Loss: 0.2417, Accuracy: 0.9278
Epoch [4/5], Loss: 0.1934, Accuracy: 0.9412
Epoch [5/5], Loss: 0.1708, Accuracy: 0.9474
sending data to divider


2023-07-07 16:41:57,992 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 2
2023-07-07 16:41:57,992 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 2
INFO:Worker_50151:Running the worker with id: 1 on iteration: 2
2023-07-07 16:41:57,996 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
2023-07-07 16:41:57,997 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 2
2023-07-07 16:41:57,996 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
2023-07-07 16:41:57,997 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 2
INFO:Worker_50152:Running the worker with id: 2 on iteration: 2
INFO:Worker_50153:Running the worker with id: 3 on iteration: 2


Epoch [1/5], Loss: 0.1944, Accuracy: 0.9416
Epoch [1/5], Loss: 0.1810, Accuracy: 0.9444
Epoch [1/5], Loss: 0.1864, Accuracy: 0.9443
Epoch [2/5], Loss: 0.1592, Accuracy: 0.9514
Epoch [2/5], Loss: 0.1514, Accuracy: 0.9549
Epoch [2/5], Loss: 0.1537, Accuracy: 0.9542
Epoch [3/5], Loss: 0.1428, Accuracy: 0.9552
Epoch [3/5], Loss: 0.1322, Accuracy: 0.9587
Epoch [3/5], Loss: 0.1316, Accuracy: 0.9597
Epoch [4/5], Loss: 0.1198, Accuracy: 0.9631
Epoch [4/5], Loss: 0.1153, Accuracy: 0.9641
Epoch [4/5], Loss: 0.1155, Accuracy: 0.9649


2023-07-07 16:42:05,444 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 3
2023-07-07 16:42:05,444 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 3
2023-07-07 16:42:05,446 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 3
INFO:Worker_50152:Running the worker with id: 2 on iteration: 3
2023-07-07 16:42:05,446 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 3
INFO:Worker_50153:Running the worker with id: 3 on iteration: 3


Epoch [5/5], Loss: 0.1066, Accuracy: 0.9670
Epoch [5/5], Loss: 0.1004, Accuracy: 0.9677
sending data to divider
Epoch [5/5], Loss: 0.1029, Accuracy: 0.9666
sending data to divider
sending data to divider
Epoch [1/5], Loss: 0.1212, Accuracy: 0.9638
Epoch [1/5], Loss: 0.1180, Accuracy: 0.9634
Epoch [2/5], Loss: 0.1036, Accuracy: 0.9666
Epoch [2/5], Loss: 0.1030, Accuracy: 0.9673
Epoch [3/5], Loss: 0.0969, Accuracy: 0.9703
Epoch [3/5], Loss: 0.0907, Accuracy: 0.9706
Epoch [4/5], Loss: 0.0788, Accuracy: 0.9738
Epoch [4/5], Loss: 0.0850, Accuracy: 0.9719
Epoch [5/5], Loss: 0.0723, Accuracy: 0.9765
Epoch [5/5], Loss: 0.0756, Accuracy: 0.9751
sending data to divider
sending data to divider
